### Examples Of Pyspark ML

#### INICIAMOS SESION

In [1]:
import os,sys
from pathlib import Path
# JAVA_HOME debe señalar un JDK completo (Java 21), no un runtime reducido.
assert os.environ.get('JAVA_HOME'), 'Configura JAVA_HOME con el JDK 21 completo'
os.environ['PYSPARK_PYTHON']=sys.executable
os.environ['SPARK_LOCAL_IP']='127.0.0.1'
from pyspark.sql import SparkSession,functions as F
spark=(SparkSession.builder.master('local[2]').appName('Unidades siguientes')
 .config('spark.driver.bindAddress','127.0.0.1').config('spark.sql.shuffle.partitions','4')
 .config('spark.ui.enabled','false').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark',spark.version,'Java',os.environ['JAVA_HOME'])

C:\Users\Usuario\Documents\Codex\2026-09-24\va\work\venv-next\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark 4.2.0 Java C:\Users\Usuario\Documents\Codex\2026-09-24\va\work\java21\jdk-21.0.12.1+1


CARGAMOS CSV

In [2]:
training = spark.read.csv('data/test1.csv', header=True, inferSchema=True)

In [3]:
training.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



En PySpark se trabaja de forma diferente. Tendremos que agrupar nuestras variables independientes de forma que queden todas en una columna y dentro de una lista, por lo que crearemos un vector de ensamblaje o "vector assembler", de tal modo que queden así esas variables independientes:
- [Age, Experience]

Lo que haremos con estas dos, será tratarlas como una nueva variable independiente:
- [Age, Experience] ----> nueva_variable_independiente

In [4]:
from pyspark.ml.feature import VectorAssembler 

In [5]:
feature_assembler = VectorAssembler(inputCols=['age', 'Experience'], outputCol='Independent features') # 1:46:27 del vídeo

In [6]:
output = feature_assembler.transform(training)

Veremos que se crea una nueva columna cuyos valores se corresponden a unos array con el contenido de aquellas variables independientes que hemos agrupado. Esto será nuestro input feature o lo que solíamos definir como train.

In [7]:
output.show()

+---------+---+----------+------+--------------------+
|     Name|age|Experience|Salary|Independent features|
+---------+---+----------+------+--------------------+
|    Krish| 31|        10| 30000|         [31.0,10.0]|
|Sudhanshu| 30|         8| 25000|          [30.0,8.0]|
|    Sunny| 29|         4| 20000|          [29.0,4.0]|
|     Paul| 24|         3| 20000|          [24.0,3.0]|
|   Harsha| 21|         1| 15000|          [21.0,1.0]|
|  Shubham| 23|         2| 18000|          [23.0,2.0]|
+---------+---+----------+------+--------------------+



Seleccionamos las columnas que nos interesan para nuestro modelo: el train (Independent Features) y el test (Salary)

In [8]:
finalized_data = output.select('Independent features', 'Salary')
finalized_data.show()

+--------------------+------+
|Independent features|Salary|
+--------------------+------+
|         [31.0,10.0]| 30000|
|          [30.0,8.0]| 25000|
|          [29.0,4.0]| 20000|
|          [24.0,3.0]| 20000|
|          [21.0,1.0]| 15000|
|          [23.0,2.0]| 18000|
+--------------------+------+



A continuación, entrenaremos un modelo de regresión lineal.

In [9]:
from pyspark.ml.regression import LinearRegression

In [10]:
(train,test)=finalized_data.randomSplit([.75,.25],seed=42)
assert train.count()>0 and test.count()>0
regressor=LinearRegression(featuresCol='Independent features',labelCol='Salary').fit(train)

In [11]:
regressor.coefficients

DenseVector([-64.8464, 1584.7554])

In [12]:
regressor.intercept

15414.10693970376

In [13]:
prediction = regressor.evaluate(test)

In [14]:
prediction.predictions.show()

+--------------------+------+------------------+
|Independent features|Salary|        prediction|
+--------------------+------+------------------+
|          [24.0,3.0]| 20000|18612.059158134223|
+--------------------+------+------------------+



In [15]:
# Errores

prediction.meanAbsoluteError, prediction.meanSquaredError

(1387.9408418657767, 1926379.780519081)

In [16]:
spark.stop()